In [1]:
import polars as pl

In [4]:
!pip install xlsx2csv

ERROR: Could not find an activated virtualenv (required).


In [4]:
import polars as pl
df = pl.read_excel("Pacotes.xlsx")
print(df)

shape: (47_777, 34)
┌───────────┬───────────┬───────────┬───────────┬───┬───────────┬───────────┬───────────┬──────────┐
│ Atualizad ┆ _duplicat ┆ _duplicat ┆ _duplicat ┆ … ┆ _duplicat ┆ _duplicat ┆ _duplicat ┆ _duplica │
│ o em 2026 ┆ ed_0      ┆ ed_1      ┆ ed_2      ┆   ┆ ed_29     ┆ ed_30     ┆ ed_31     ┆ ted_32   │
│ -01-14    ┆ ---       ┆ ---       ┆ ---       ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---      │
│ ---       ┆ str       ┆ str       ┆ str       ┆   ┆ str       ┆ str       ┆ str       ┆ str      │
│ str       ┆           ┆           ┆           ┆   ┆           ┆           ┆           ┆          │
╞═══════════╪═══════════╪═══════════╪═══════════╪═══╪═══════════╪═══════════╪═══════════╪══════════╡
│ Dados     ┆ null      ┆ null      ┆ Referênci ┆ … ┆ null      ┆ null      ┆ null      ┆ null     │
│ Básicos   ┆           ┆           ┆ as        ┆   ┆           ┆           ┆           ┆          │
│           ┆           ┆           ┆ Externas  ┆   ┆           ┆      

In [16]:
!echo "id os group desc date class un field_number field_location depth suit shipment company resource vessel relatbar n1710 files_count files_size files_missing_count files_missing_size files_oldest files_newest inventory_first inventory_newest files_missing_lastseen bucket base folder path" | tr ' ' '\n' \
| sed -r 's/^(.*)$/  "\1" : pl.Utf8,/'

  "id" : pl.Utf8,
  "os" : pl.Utf8,
  "group" : pl.Utf8,
  "desc" : pl.Utf8,
  "date" : pl.Utf8,
  "class" : pl.Utf8,
  "un" : pl.Utf8,
  "field_number" : pl.Utf8,
  "field_location" : pl.Utf8,
  "depth" : pl.Utf8,
  "suit" : pl.Utf8,
  "shipment" : pl.Utf8,
  "company" : pl.Utf8,
  "resource" : pl.Utf8,
  "vessel" : pl.Utf8,
  "relatbar" : pl.Utf8,
  "n1710" : pl.Utf8,
  "files_count" : pl.Utf8,
  "files_size" : pl.Utf8,
  "files_missing_count" : pl.Utf8,
  "files_missing_size" : pl.Utf8,
  "files_oldest" : pl.Utf8,
  "files_newest" : pl.Utf8,
  "inventory_first" : pl.Utf8,
  "inventory_newest" : pl.Utf8,
  "files_missing_lastseen" : pl.Utf8,
  "bucket" : pl.Utf8,
  "base" : pl.Utf8,
  "folder" : pl.Utf8,
  "path" : pl.Utf8,


In [8]:
import polars as pl

# ============================================
# Define schema based on your description
# ============================================

schema = {
  "id"     : pl.Int32,

  "os"     : pl.Categorical,
  "group"  : pl.Categorical,
  "desc"   : pl.Utf8,
  "date"   : pl.Date,
  "class"  : pl.Utf8,
  "un"     : pl.Categorical,

  "field_number"   : pl.Int16,
  "field_location" : pl.Categorical,
  "depth"          : pl.Int32, # could not parse `87693` as dtype `i16` at column 'depth'

  "suit"     : pl.Utf8,
  "shipment" : pl.Utf8,

  "company"  : pl.Categorical,
  "resource" : pl.Categorical,
  "vessel"   : pl.Categorical,
  "relatbar" : pl.Utf8,
  "n1710"    : pl.Utf8,

  "files_count" : pl.Int64,
  "files_size"  : pl.Float64,
  "files_missing_count" : pl.Int64,
  "files_missing_size"  : pl.Float64,
  "files_oldest"     : pl.Date,
  "files_newest"     : pl.Date,
  "inventory_first"  : pl.Date,
  "inventory_newest" : pl.Date,
  "files_missing_lastseen" : pl.Date,

  "bucket" : pl.Categorical,
  "base"   : pl.Utf8,
  "folder" : pl.Utf8,
  "path"   : pl.Utf8,
}

# ============================================
# Load CSV with forced schema
# ============================================
df = pl.read_excel(
  'dados-submarinos.xlsx',
  sheet_name = None,
  schema_overrides = schema,
  infer_schema_length = None
)

# ============================================
# 
# ============================================
for col, dtype in df.schema.items():
  if dtype == pl.Categorical:
    phys_name  = f'{col}_physical'
    count_name = f'{col}_count'
    df = df.with_columns(
      pl.col( col )
        .to_physical( )
        .alias( phys_name ),
      pl.col( col )
        .count( )
        .over( col )
        .alias( count_name )
    )

    # rearrange so the physical column is placed immediately after the original
    cols = df.columns
    
    cols.remove( count_name )
    cols.insert(
      cols.index( col ) + 1, count_name
    )
    
    cols.remove( phys_name )
    cols.insert(
      cols.index( col ) + 1, phys_name
    )
    
    df = df.select( cols )


# ============================================
# 
# ============================================
df = df.with_columns(
  pl.col( 'company' )
    .fill_null(
      pl.col( 'company' )
      .forward_fill( )
      .over( 'vessel' )
    )
)

# ============================================
# Save as Parquet
# ============================================
df.write_parquet("dados-submarinos-all.parquet")

print('XLSX successfully converted to Parquet with correct data types!')

print( df.dtypes )
df.head( )

ModuleNotFoundError: required package 'xlsx2csv' not found.
Please install using the command `pip install xlsx2csv`.

In [58]:
df = pl.read_parquet( 'dados-submarinos-pacotes.parquet' )
df
#df = df.with_columns(
#  pl.col( 'depth' ).cast( pl.Int32 )
#)
#df.write_parquet("dados-submarinos.parquet")

id,os,os_physical,group,group_physical,desc,date,class,un,un_physical,field_number,field_location,field_location_physical,depth,suit,shipment,company,company_physical,resource,resource_physical,vessel,vessel_physical,relatbar,n1710,files_count,files_size,files_missing_count,files_missing_size,files_oldest,files_newest,inventory_first,inventory_newest,files_missing_lastseen,bucket,bucket_physical,base,folder,path
i32,cat,u32,cat,u32,str,date,str,cat,u32,i16,cat,u32,i32,str,str,cat,u32,cat,u32,cat,u32,str,str,i64,f64,i64,f64,date,date,date,date,date,cat,u32,str,str,str
112432,"""6000792398""",0,"""ANC""",3297,"""Hook OUT linha #7 para estalei…",2025-11-26,"""Hook OUT""","""UN-BC - SEDE Bacia""",3315,312,"""7-CG-5HP-RJS""",3324,283,"""7-CG-5HP-RJS""","""Abandonos 2022""","""C-Innovation""",3902,"""AHTS""",3918,"""Bram Titan""",3928,"""BTCG26-004""",null,8,4.1596e9,0,0.0,2026-01-13,2026-01-13,2026-01-14,2026-01-14,null,"""sistemas-ancoragem""",4008,"""recursos-embarcacoes""","""OS006000792398""","""sistemas-ancoragem/recursos-em…"
112431,"""6000797937""",1,null,null,null,null,null,null,null,null,null,null,null,null,null,"""AKOFS""",3903,"""MPSV""",3919,"""Akofs Santos""",3929,null,null,34,2.3224e10,0,0.0,2026-01-13,2026-01-14,2026-01-14,2026-01-14,null,"""servicos-arvore-de-natal-molha…",4009,"""recursos-embarcacoes""","""OS006000797937""","""servicos-arvore-de-natal-molha…"
112430,"""6000797560""",2,null,null,null,null,null,null,null,null,null,null,null,null,null,"""AKOFS""",3903,"""MPSV""",3919,"""Akofs Santos""",3929,"""PDBATP22-006""",null,17,5.9127e9,0,0.0,2026-01-13,2026-01-13,2026-01-14,2026-01-14,null,"""servicos-arvore-de-natal-molha…",4009,"""recursos-embarcacoes""","""OS006000797560""","""servicos-arvore-de-natal-molha…"
112429,"""6000789371""",3,"""MIS""",3298,"""Inspecao complementar em dutos…",2025-10-29,"""Inspeção complementar em dutos…","""UN-ES - SEDE Espíri""",3316,881,"""P-52""",3325,1770,"""Eor P-52""","""Eor ESDV-1 P-52""","""Oceânica""",3904,"""SDSV""",3920,"""Oceanica Subiv""",3930,null,null,11,3.1623e9,0,0.0,2026-01-13,2026-01-13,2026-01-14,2026-01-14,null,"""manutencao-inspecao-submarina""",4010,"""recursos-embarcacoes""","""OS006000789371-P-52-INSP-DUTO-…","""manutencao-inspecao-submarina/…"
112428,"""6000749623""",4,"""MER""",3299,"""Pos PULL-IN de duto flexivel (…",2024-12-17,"""Pós PULL-IN de duto flexível""","""UN-BS - EDISA II Sa""",3317,120,"""8-LL-118D-RJS""",3326,2115,"""Reinterligacao IA LL-118""","""Reinterligacao LL-118""","""Oceânica""",3904,"""SDSV""",3920,"""Oceanica Subiv""",3930,null,null,8,7.81365368e8,0,0.0,2026-01-13,2026-01-13,2026-01-14,2026-01-14,null,"""manutencao-inspecao-submarina""",4010,"""recursos-embarcacoes""","""OS006000749623-P-67-POS-PULL-I…","""manutencao-inspecao-submarina/…"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
4178,"""6000487025""",23557,"""EQS""",3305,"""Insp. dif. - med. de espes. 1-…",2018-03-13,"""Insp. dif. - med. de espes.""","""UN-BC - SEDE Bacia""",3315,347,"""1-RJS-414""",7507,129,"""ou BC / ATP - N""","""Inspecoes (ICPI) UO-BC 2018""","""C-Innovation""",3902,"""RSV""",3921,"""Joe Griffin""",3976,"""JGCO21-014""",null,30,1.7990e10,52,2.7140e10,2021-02-26,2024-09-10,2021-02-14,2024-09-15,2024-09-15,"""manutencao-inspecao-submarina""",4010,"""recursos-embarcacoes""","""OS006000487025-JGCO21-014""","""manutencao-inspecao-submarina/…"
4177,"""6000486667""",23558,"""EQS""",3305,"""Insp. dif. - med. de espes. 9-…",2018-03-09,"""Insp. dif. - med. de espes.""","""UN-BC - SEDE Bacia""",3315,247,"""7-CRT-17HP-RJS""",20449,1080,"""ATP-BRC""","""Inspecoes (ICPI) UO-RIO 2018""","""C-Innovation""",3902,"""RSV""",3921,"""Joe Griffin""",3976,"""JGCRT21-020""",null,51,2.7116e10,93,5.2230e10,2021-02-26,2024-09-10,2021-02-14,2024-09-15,2024-09-15,"""manutencao-inspecao-submarina""",4010,"""recursos-embarcacoes""","""OS006000486667-JGCRT21-020""","""manutencao-inspecao-submarina/…"
4107,"""6000558142""",23559,"""EQS""",3305,"""Insp. dif. - 

In [4]:
!conda env list

/bin/bash: line 1: conda: command not found


In [10]:
pip install xlsx2csv

ERROR: Could not find an activated virtualenv (required).
Note: you may need to restart the kernel to use updated packages.


In [3]:
!rm -rf lib

In [12]:
!jupyter kernelspec list

Available kernels:
  ir         /usr/local/share/jupyter/kernels/ir
  python3    /usr/local/share/jupyter/kernels/python3


In [5]:
!export PATH="$HOME/miniforge3/bin:$PATH"

In [6]:
!conda env list

/bin/bash: line 1: conda: command not found
